# 07 — Video Inference: ByteTrack + Live Overlay + QC Decision Logic · Banana Ripeness (YOLO26s)

**หน้าที่ของ notebook นี้ (MASTER PROMPT ข้อ 10)**
1. รับ input ได้ 4 แบบ: **ไฟล์วิดีโอ / โฟลเดอร์ภาพ / webcam / RTSP** (ตั้งที่ `SOURCE` ใน CELL 1)
2. **วิดีโอจำลอง** (ตกลง 2026-09-20): สร้างสายพานจำลองจากภาพ test — ภาพเลื่อนผ่านเฟรมเหมือนกล้วยบนสายพาน → มี ground truth ว่าควรนับได้กี่ผลต่อคลาส
3. **ByteTrack** (`tracker="bytetrack.yaml"` — ต้องระบุ เพราะ default ของ ultralytics 8.4 คือ tracktrack) → นับกล้วยผลเดิมครั้งเดียว, คลาสของแต่ละ track = majority vote ตลอดช่วงที่เห็น
   - **v3 (2026-09-20):** รอบแรก nms=False นับเกิน +10/40 ทั้งหมดบนภาพที่กล้วยซ้อนกัน → CELL 3 ทำ ablation `nms × conf × min_track_frames` บนวิดีโอจำลอง (มี ground truth) แล้วเลือก config ที่ error ต่ำสุดอัตโนมัติ (ผลจริง: NMS-free ดีกว่า NMS, min_track_frames คือตัวแปรหลัก)
   - **v3.2:** พบกรณีกล้วยที่โผล่จากขอบขวาครึ่งลูกได้ track หนึ่ง (ทายจากปลายกล้วย = semi_ripe) แล้ว track หลุดเป็นอีก id → นับ 2 ครั้ง → เพิ่มกฎ **`cross_line`** (นับเมื่อจุดกลางเคลื่อนผ่านเส้นกลางเฟรม) + โหวตคลาสเฉพาะกล่องที่ไม่ชิดขอบ; ablation เทียบ `cross_line` vs `min_frames`
4. วาด bbox + label **ไทย/อังกฤษ** + confidence + สีประจำคลาส (unripe เขียว / semi_ripe เหลืองอมเขียว / ripe เหลือง / overripe น้ำตาล) ด้วยฟอนต์ Sarabun
5. Overlay สด: นับสะสมต่อคลาส, % สัดส่วน, FPS, สถานะ QC ของผลิตภัณฑ์ที่เลือก + **แจ้งเตือนเมื่อ overripe เกิน threshold**
6. Export: `videos/output/<name>_annotated.mp4`, `reports/detection_log.csv` (frame, timestamp, track_id, class, confidence, bbox), `reports/summary_report.csv`, `figures/07_*.png`

> kernel: **ML Environment (ds69) + YOLO26** · ใช้ `final_model` จาก nb06 (`configs/project.yaml`)


In [ ]:
# === CELL 1 === โหลด config + โมเดลสุดท้าย + ฟอนต์ไทย + ตั้งค่า SOURCE / QC / tracking ไว้ที่เดียว
# ทำอะไร : อ่าน final_model (best.pt, conf, nms=False, imgsz) จาก project.yaml, โหลดฟอนต์ Sarabun (ไทย+อังกฤษ) ลง fonts/,
#          กำหนด QC profile ต่อผลิตภัณฑ์ + เกณฑ์แจ้งเตือน overripe + แหล่งวิดีโอ
# ทำไม   : spec ข้อ 10 — ค่าทุกอย่างที่ผู้ใช้โรงงานต้องปรับ (ผลิตภัณฑ์, threshold, กล้อง) ต้องอยู่ที่เดียว ไม่ฝังในโค้ด
import os, sys, json, time, shutil, random, importlib, math, csv, urllib.request
from pathlib import Path
from collections import Counter, defaultdict, deque
from datetime import datetime
import yaml
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().resolve()
CFG_PATH = PROJECT_ROOT / "configs" / "project.yaml"
with open(CFG_PATH, encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)
DIRS = {k: PROJECT_ROOT / v for k, v in CONFIG["dirs"].items()}
SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)
CLASSES = {int(k): v for k, v in CONFIG["classes"].items()}
NAMES = list(CLASSES.values())
NC = len(CLASSES)
TH = {int(k): v for k, v in CONFIG["classes_th"].items()}
RGB = {int(k): tuple(v) for k, v in CONFIG["class_colors_rgb"].items()}
BGR = {k: (v[2], v[1], v[0]) for k, v in RGB.items()}
sys.path.insert(0, str(DIRS["src"]))
import viz
importlib.reload(viz)
viz.apply_style()
assert "final_model" in CONFIG, "ไม่พบ final_model ใน project.yaml → รัน 06_evaluate.ipynb ให้จบก่อน"
FM = CONFIG["final_model"]
WEIGHTS = PROJECT_ROOT / FM["best"]
assert WEIGHTS.exists(), f"ไม่พบ {WEIGHTS}"
DEVICE = 0 if torch.cuda.is_available() else "cpu"
QUICK_TEST = os.environ.get("NB07_QUICK_TEST", "0") == "1"

# --- เลือกแหล่งวิดีโอตอนรัน: cell นี้จะแสดงรายการไฟล์ใน videos/input/ แล้วมีช่องให้พิมพ์ใต้ cell (Enter = วิดีโอจำลอง) ---
#   - อัปโหลดวิดีโอของตัวเองไว้ที่ videos/input/ ก่อน (JupyterLab: เข้าโฟลเดอร์ → ปุ่ม Upload) แล้วรัน cell นี้ใหม่จะเห็นในรายการ
#   - พิมพ์หมายเลข / ชื่อไฟล์ / path เต็ม / rtsp://... ก็ได้  → ถ้าไม่ใช่วิดีโอจำลอง จะถามต่อว่ากล้วยเคลื่อนที่ทางไหน (กำหนดกฎนับ)
#   - ไม่มี ground truth → CELL 2/3 ข้ามวิดีโอจำลอง + ablation → ใช้ค่า nms / min_track_frames ใน NB07 ด้านล่างโดยตรง
#   - รันแบบไม่โต้ตอบ (สคริปต์/ทดสอบ): ตั้ง env NB07_SOURCE=<path หรือ sim> [+ NB07_MOTION=1|2|3]
SIM_VIDEO = DIRS["videos_in"] / "simulated_conveyor.mp4"
VIDEO_EXT = {".mp4", ".avi", ".mov", ".mkv", ".m4v", ".webm"}
MOTION = {"1": ("x", "cross_line", "ซ้าย↔ขวา (สายพานทั่วไป) → นับเมื่อผ่านเส้นตั้งกลางเฟรม"),
          "2": ("y", "cross_line", "บน↔ล่าง (เช่น มือถือแนวตั้งมองสายพานจากบน) → นับเมื่อผ่านเส้นนอนกลางเฟรม"),
          "3": ("x", "min_frames", "อยู่นิ่ง / กล้องแพน → นับเมื่อเห็นครบ min_track_frames")}
def ask(prompt, default=""):
    try:
        return (input(prompt).strip() or default)
    except Exception:                      # ไม่มี stdin (nbconvert / สคริปต์) → ใช้ค่า default
        return default
def resolve_video(ans):
    # รับได้ทั้ง: ชื่อไฟล์ใน videos/input/ · path เต็ม · path ที่ copy มาจาก file browser (เทียบด้วยชื่อไฟล์ท้ายสุด) · rtsp://
    ans = ans.strip().strip('"').strip("'")
    if ans.lower().startswith(("rtsp://", "rtmp://", "http://", "https://")):
        return ans
    name = Path(ans).name
    for c in (Path(ans), DIRS["videos_in"] / ans, PROJECT_ROOT / ans, Path.home() / ans, Path.home() / "notebooks" / ans, DIRS["videos_in"] / name):
        if c.is_file():
            return str(c.resolve())
    return None
def choose_source():
    env_src = os.environ.get("NB07_SOURCE")
    if QUICK_TEST and not env_src:
        return str(SIM_VIDEO), *MOTION["1"][:2]
    files = sorted(p for p in DIRS["videos_in"].glob("*") if p.suffix.lower() in VIDEO_EXT and p.name != SIM_VIDEO.name)
    if env_src is None:
        print("วิดีโอที่มีใน videos/input/:")
        print(f"  [0] {SIM_VIDEO.name}  ← วิดีโอจำลองจากภาพ test (มี ground truth; CELL 2 สร้างให้ถ้ายังไม่มี)")
        for i, p in enumerate(files, 1):
            print(f"  [{i}] {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")
        if not files:
            print("  (ยังไม่มีวิดีโออื่น — อัปโหลดไว้ที่ videos/input/ แล้วรัน cell นี้ใหม่)")
    src = None
    for attempt in range(3):
        ans = env_src if env_src is not None else ask("พิมพ์หมายเลข [0-{}] แล้วกด Enter (Enter เปล่า = 0 วิดีโอจำลอง): ".format(len(files)), "0")
        if ans in ("0", "sim") or Path(ans).name == SIM_VIDEO.name:
            return str(SIM_VIDEO), *MOTION["1"][:2]
        if ans.isdigit() and 1 <= int(ans) <= len(files):
            src = str(files[int(ans) - 1])
        else:
            src = resolve_video(ans)
        if src:
            break
        assert env_src is None, f"NB07_SOURCE={env_src} ไม่พบไฟล์"
        print(f"  [!] ไม่พบ '{ans}' — พิมพ์แค่หมายเลขในรายการด้านบน หรือชื่อไฟล์ที่อยู่ใน videos/input/ (เช่น banana_line.mp4)")
    if not src:
        print("  [!] ใช้วิดีโอจำลองแทน")
        return str(SIM_VIDEO), *MOTION["1"][:2]
    m = os.environ.get("NB07_MOTION")
    if m is None:
        print("\nกล้วยในวิดีโอนี้เคลื่อนที่อย่างไร?")
        for k, v in MOTION.items():
            print(f"  [{k}] {v[2]}")
        m = ask("พิมพ์หมายเลข [1-3] แล้วกด Enter (Enter เปล่า = 1): ", "1")
    axis, rule, _ = MOTION.get(m.strip(), MOTION["1"])
    return src, axis, rule
SOURCE, LINE_AXIS, COUNT_RULE = choose_source()

NB07 = {
    "model": {"weights": str(WEIGHTS), "imgsz": int(FM.get("imgsz", 640)), "conf": float(FM["conf"]), "iou": float(FM.get("iou", 0.7)),
              "nms": False,                      # False = หัว NMS-free (ablation v3 รอบจริง: ดีกว่า NMS ปกติทุกจุด) | None = NMS ปกติ IoU 0.7
              #                                    ค่านี้ใช้ตรง ๆ เมื่อ SOURCE เป็นวิดีโอจริง (ไม่มี ablation); กับวิดีโอจำลอง CELL 3 เลือกให้อีกที
              "tracker": CONFIG["model"].get("tracker", "bytetrack.yaml")},   # ByteTrack (default ของ ultralytics 8.4 คือ tracktrack → ต้องระบุ)
    "sim": {"n_images": 30, "fps": 30, "canvas": (1280, 720), "speed_px": 16, "gap_px": 220, "max_h": 560},   # วิดีโอจำลอง
    "count": {
        "rule": COUNT_RULE,                  # (จากคำถามด้านบน) "cross_line" = นับเมื่อจุดกลาง track เคลื่อนผ่านเส้นนับ — ผลที่เข้ามาครึ่งลูกที่ขอบแล้ว track หลุดจะไม่ถูกนับ
        #                                      "min_frames" = นับเมื่อเห็นครบเฟรมอย่างเดียว (กล้องนิ่ง / ผลไม่เคลื่อนที่) — CELL 3 เลือกจาก ablation
        "line_axis": LINE_AXIS,              # (จากคำถามด้านบน) "x" = เส้นตั้ง (ผลเคลื่อนซ้าย↔ขวา) | "y" = เส้นนอน (ผลเคลื่อนบน↔ล่าง)
        "line_x": 0.5,                       # ตำแหน่งเส้นนับ (สัดส่วนความกว้าง ถ้า axis x / ความสูง ถ้า axis y)
        "min_track_frames": 5,               # track ต้องถูกเห็น ≥ N เฟรมจึงนับ (กัน detection วูบวาบ/กล่องซ้ำชั่วคราว) — ใช้ร่วมกับ rule
        "edge_margin": 0.02,                 # กล่องที่ชิดขอบซ้าย/ขวาเกินสัดส่วนนี้ = เห็นไม่เต็มลูก → ไม่ใช้โหวตคลาส (ปลายกล้วยที่โผล่มาก่อนมักดูเขียวกว่าจริง)
    },
    "ablation": {                            # CELL 3: ทดลองนับ (ไม่วาด) บนวิดีโอจำลองที่รู้คำตอบ แล้วเลือก config ที่ error ต่ำสุด
        "enabled": True, "auto_select": True,
        "nms": [False, None],                # NMS-free vs NMS ปกติ (v3 รอบจริง: NMS-free ดีกว่าทุกจุด)
        "conf": [float(FM["conf"])],         # v3 รอบจริง: 0.10 กับ conf ของ final_model ให้ผลเท่ากัน → เหลือค่าเดียว (ใส่เพิ่มใน list ได้)
        "rule": ["cross_line", "min_frames"],   # กฎการนับ (ประเมิน offline จาก track ชุดเดียวกัน — ไม่ต้องรัน tracking เพิ่ม)
        "min_track_frames": [3, 5, 8, 12, 16],   # error ลดลงตามค่านี้ (v3 รอบจริง: 10 → 5 → 3) → ขยาย grid ดูว่าถึงไหนแล้วไม่ลดต่อ
    },
    "qc": {
        "products": {   # คลาสที่ "ผ่าน" ของแต่ละผลิตภัณฑ์ — ปรับตามเกณฑ์โรงงาน
            "fresh_pack":   {"pass": ["ripe"],                "th": "กล้วยหอมแพ็คสด (ต้องสุกพอดี)"},
            "banana_chips": {"pass": ["unripe", "semi_ripe"], "th": "กล้วยฉาบ (ใช้ดิบ/ห่าม)"},
            "puree_bakery": {"pass": ["ripe", "overripe"],    "th": "กล้วยบดเบเกอรี่ (สุกถึงงอม)"},
        },
        "active_product": "fresh_pack",
        "overripe_alert_share": 0.20,        # แจ้งเตือนเมื่อสัดส่วน overripe ใน window ล่าสุด > 20%
        "overripe_alert_window": 20,         # นับจาก 20 ผลล่าสุดที่ถูกยืนยัน
        "alert_min_count": 5,                # ต้องมีผลยืนยันแล้วอย่างน้อย 5 ผลก่อนเริ่มเตือน
    },
    "overlay": {"font_size": 18, "box_font_size": 17, "panel_alpha": 0.6,    # ขนาดที่ความสูงเฟรม 720 px
                "ref_height": 720, "scale": 1.0,                             # ฟอนต์/เส้น/panel ย่อ-ขยายตามความสูงเฟรมจริงอัตโนมัติ; scale = ปรับเพิ่มเอง (0.7 = เล็กลง 30%)
                "panel": True},                                              # False = ซ่อน panel สถิติ (เหลือแค่กล่อง+label)
}
if QUICK_TEST:
    NB07["sim"].update(n_images=6, speed_px=28)
    NB07["model"]["imgsz"] = 320
    NB07["ablation"].update(min_track_frames=[3, 5])

# --- ฟอนต์ไทย+อังกฤษ (Sarabun, OFL) → fonts/ ; fallback: ฟอนต์ระบบที่รองรับไทย ; fallback สุดท้าย: default (แสดงอังกฤษอย่างเดียว) ---
FONT_DIR = PROJECT_ROOT / "fonts"
FONT_DIR.mkdir(exist_ok=True)
FONT_PATH = FONT_DIR / "Sarabun-Regular.ttf"
FONT_URL = "https://raw.githubusercontent.com/google/fonts/main/ofl/sarabun/Sarabun-Regular.ttf"
THAI_OK = True
if not FONT_PATH.exists():
    try:
        urllib.request.urlretrieve(FONT_URL, FONT_PATH)
        print(f"[OK] ดาวน์โหลดฟอนต์ Sarabun → {FONT_PATH.relative_to(PROJECT_ROOT)} ({FONT_PATH.stat().st_size // 1024} KB)")
    except Exception as e:
        print(f"[WARN] ดาวน์โหลดฟอนต์ไม่ได้ ({e}) → หาฟอนต์ไทยในระบบ")
        import subprocess
        try:
            out = subprocess.run(["fc-list", ":lang=th", "file"], capture_output=True, text=True, timeout=10).stdout
            cands = [l.split(":")[0].strip() for l in out.splitlines() if l.strip().lower().endswith((".ttf", ".otf"))]
            if cands:
                FONT_PATH = Path(cands[0])
                print(f"[OK] ใช้ฟอนต์ระบบ {FONT_PATH}")
            else:
                THAI_OK = False
        except Exception:
            THAI_OK = False
def load_font(size):
    if THAI_OK and Path(FONT_PATH).exists():
        return ImageFont.truetype(str(FONT_PATH), size)
    return ImageFont.load_default(size=size)
FONT, FONT_BOX, FONT_BIG = load_font(NB07["overlay"]["font_size"]), load_font(NB07["overlay"]["box_font_size"]), load_font(30)
if not THAI_OK:
    print("[WARN] ไม่มีฟอนต์ไทย → label จะแสดงภาษาอังกฤษอย่างเดียว")

MODEL = YOLO(str(WEIGHTS))
RUN = {"notebook": "07_video_inference", "started_at": datetime.now().isoformat(timespec="seconds"), "settings": NB07, "source": str(SOURCE)}
print(f"[OK] model = {FM['name']} ({WEIGHTS.relative_to(PROJECT_ROOT)}) | conf {NB07['model']['conf']} | nms={NB07['model']['nms']} | imgsz {NB07['model']['imgsz']} | tracker {NB07['model']['tracker']} | device {DEVICE!r}")
print(f"[OK] QC product = {NB07['qc']['active_product']} ({NB07['qc']['products'][NB07['qc']['active_product']]['th']}) → ผ่าน: {NB07['qc']['products'][NB07['qc']['active_product']]['pass']} | "
      f"เตือนเมื่อ overripe > {NB07['qc']['overripe_alert_share'] * 100:.0f}% ใน {NB07['qc']['overripe_alert_window']} ผลล่าสุด")
print(f"[OK] SOURCE = {SOURCE} | กฎนับ {COUNT_RULE}" + (f" (แกน {LINE_AXIS})" if COUNT_RULE == "cross_line" else "")
      + ("  ← วิดีโอจำลอง" if str(SOURCE) == str(SIM_VIDEO) else "  ← วิดีโอภายนอก (ไม่มี ground truth)"))


In [ ]:
# === CELL 2 === สร้างวิดีโอจำลอง "สายพาน" จากภาพ test (ถ้า SOURCE คือไฟล์จำลองและยังไม่มี) + ground truth จำนวนผลต่อคลาส
# ทำอะไร : สุ่มภาพ test แบบ stratified ตามคลาสหลัก → วางเรียงบนสายพานเลื่อนซ้ายด้วยความเร็วคงที่ → เขียน .mp4 (30 fps)
#          บันทึกจำนวน bbox จริงต่อคลาสของภาพที่ใช้ = จำนวนที่ระบบ "ควรนับได้" ไว้เทียบใน CELL 4
# ทำไม   : ยังไม่มีวิดีโอจริง (ตกลง 2026-09-20) — จำลองให้ทดสอบ tracking/นับ/QC ได้ครบ และวัด counting error ได้เพราะรู้คำตอบ
SIM = NB07["sim"]
sim_path = DIRS["videos_in"] / "simulated_conveyor.mp4"
sim_meta = DIRS["videos_in"] / "simulated_conveyor.json"
is_sim = str(SOURCE) == str(sim_path)
EXPECTED, SIM_META = None, None
def _meta_ok():
    if not (sim_path.exists() and sim_meta.exists()):
        return False
    with open(sim_meta, encoding="utf-8") as f:
        return "tiles" in json.load(f)       # json รุ่นเก่า (ไม่มีตำแหน่ง tile) → สร้างใหม่เพื่อให้ตรวจต่อภาพได้
if is_sim and not _meta_ok():
    man = pd.read_csv(DIRS["reports"] / "manifest_split.csv")
    test_df = man[man.split == "test"].copy()
    rng = random.Random(SEED)
    per_cls = max(1, SIM["n_images"] // NC)
    picks = []
    for c in CLASSES:
        pool = test_df[test_df.dominant_class == c]["file"].tolist()
        picks += rng.sample(pool, min(per_cls, len(pool)))
    rng.shuffle(picks)
    W, H = SIM["canvas"]
    tiles, expected, tiles_meta = [], Counter(), []
    src_of = dict(zip(man.file, man.source))
    for f in picks:
        img = cv2.imread(str(DIRS["datasets"] / "test" / "images" / f))
        s = min(SIM["max_h"] / img.shape[0], (W * 0.8) / img.shape[1])
        img = cv2.resize(img, (int(img.shape[1] * s), int(img.shape[0] * s)), interpolation=cv2.INTER_AREA)
        tiles.append(img)
        gt = Counter()
        for ln in (DIRS["datasets"] / "test" / "labels" / (Path(f).stem + ".txt")).read_text().splitlines():
            if ln.strip():
                gt[NAMES[int(ln.split()[0])]] += 1
        expected.update(gt)
        tiles_meta.append({"image": f, "source": src_of.get(f, "?"), "width": int(img.shape[1]), "height": int(img.shape[0]), "gt": {n: int(gt.get(n, 0)) for n in NAMES}})
    starts, x = [], W
    for t, tm in zip(tiles, tiles_meta):         # ตำแหน่งเริ่มของแต่ละภาพบนสายพาน (ต่อกันด้วยช่องว่าง gap_px)
        starts.append(x)
        tm["start"] = int(x)
        x += t.shape[1] + SIM["gap_px"]
    total_len = x
    n_frames = int(math.ceil(total_len / SIM["speed_px"]))
    writer = cv2.VideoWriter(str(sim_path), cv2.VideoWriter_fourcc(*"mp4v"), SIM["fps"], (W, H))
    for fi in range(n_frames):
        frame = np.full((H, W, 3), 70, dtype=np.uint8)               # สายพานสีเทาเข้ม
        off = (fi * SIM["speed_px"]) % 80
        for yy in (80, H - 80):                                         # ขอบสายพาน + ขีดเลื่อนให้เห็นการเคลื่อนที่
            cv2.line(frame, (0, yy), (W, yy), (110, 110, 110), 3)
        for xx in range(-off, W, 80):
            cv2.line(frame, (xx, H - 78), (xx + 30, H - 78), (140, 140, 140), 2)
        shift = fi * SIM["speed_px"]
        for t, sx in zip(tiles, starts):
            x0 = sx - shift
            th, tw = t.shape[:2]
            y0 = (H - th) // 2
            if x0 >= W or x0 + tw <= 0:
                continue
            xa, xb = max(0, x0), min(W, x0 + tw)
            frame[y0:y0 + th, xa:xb] = t[:, xa - x0:xb - x0]
        writer.write(frame)
    writer.release()
    EXPECTED = {n: int(expected.get(n, 0)) for n in NAMES}
    SIM_META = {"images": picks, "expected_counts": EXPECTED, "frames": n_frames, "fps": SIM["fps"], "canvas": [W, H], "settings": SIM, "tiles": tiles_meta}
    with open(sim_meta, "w", encoding="utf-8") as f:
        json.dump(SIM_META, f, ensure_ascii=False, indent=2)
    print(f"[OK] สร้าง {sim_path.relative_to(PROJECT_ROOT)}: {len(picks)} ภาพ test → {n_frames} เฟรม ({n_frames / SIM['fps']:.0f} s @ {SIM['fps']} fps, {W}×{H}) | ควรนับได้: {EXPECTED}")
elif is_sim:
    with open(sim_meta, encoding="utf-8") as f:
        SIM_META = json.load(f)
    EXPECTED = SIM_META["expected_counts"]
    print(f"[SKIP] มี {sim_path.relative_to(PROJECT_ROOT)} แล้ว ({SIM_META['frames']} เฟรม, {len(SIM_META['tiles'])} ภาพ) | ควรนับได้: {EXPECTED}")
else:
    print(f"[INFO] ใช้ SOURCE ภายนอก ({SOURCE}) — ไม่มี ground truth จำนวนผล; จะรายงานเฉพาะที่นับได้")
RUN["expected_counts"] = EXPECTED


In [ ]:
# === CELL 3 === Ablation การนับ (ไม่วาด): nms × conf × min_track_frames บนวิดีโอจำลอง → เลือก config ที่นับใกล้ ground truth ที่สุด
# ทำอะไร : รัน tracking ซ้ำแบบนับอย่างเดียว 1 ครั้งต่อ (nms, conf) แล้วประเมินทุกค่า min_track_frames จากอายุ track ชุดเดียวกัน
#          → ตาราง error ต่อคลาส (reports/07_counting_ablation.csv) + กราฟ → auto-select ค่าที่ |error| รวมต่ำสุดไปใช้ใน CELL 4
# ทำไม   : รอบแรก (nms=False, min 3) นับเกิน +10/40 และทั้งหมดอยู่ในภาพที่กล้วยซ้อนกัน — ตรงกับ nb06 ที่พบกล่องซ้ำของหัว NMS-free
#          บน occlusion subset (8 คู่ vs 0) → เลือกค่าจากหลักฐานบน "validation video" ที่รู้คำตอบ ไม่ใช่เดา
M = NB07["model"]
AB = NB07["ablation"]
CNT = NB07["count"]
def new_track(frame_idx):
    return {"classes": Counter(), "classes_full": Counter(), "frames": 0, "c_min": float("inf"), "c_max": float("-inf"),
            "first": frame_idx, "last": frame_idx, "confirmed_at": None}
def update_track(tr, box, c, cf, shape, frame_idx):
    # โหวตคลาสถ่วง conf; แยกโหวตเฉพาะกล่องที่ไม่ชิดขอบตามแกนการเคลื่อนที่ (เห็นเต็มลูก) + จดช่วงตำแหน่งจุดกลางกล่องบนแกนนั้นไว้ตัดสิน cross_line
    x1, y1, x2, y2 = box
    H_, W_ = shape
    a, b, L = (x1, x2, W_) if CNT["line_axis"] == "x" else (y1, y2, H_)
    m = CNT["edge_margin"] * L
    tr["classes"][c] += cf
    if a > m and b < L - m:
        tr["classes_full"][c] += cf
    cc = (a + b) / 2
    tr["c_min"], tr["c_max"] = min(tr["c_min"], cc), max(tr["c_max"], cc)
    tr["frames"] += 1
    tr["last"] = frame_idx
def track_class(tr):
    votes = tr["classes_full"] if tr["classes_full"] else tr["classes"]   # ถ้าไม่เคยเห็นเต็มลูกเลย ค่อยใช้ทุกกล่อง
    return max(votes, key=votes.get)
def track_countable(tr, rule, min_frames, shape):
    if tr["frames"] < min_frames:
        return False
    if rule == "cross_line":
        line = CNT["line_x"] * (shape[1] if CNT["line_axis"] == "x" else shape[0])
        return tr["c_min"] < line < tr["c_max"]            # จุดกลางเคยอยู่ทั้งสองฝั่งของเส้น = เคลื่อนผ่านเส้นแล้ว
    return True

def run_tracks(nms, conf, max_frames=None):
    # persist=False → ultralytics สร้าง tracker ใหม่ทุกครั้งที่เรียก (ไม่ให้ id/สถานะข้ามรอบ); เปลี่ยน nms → predictor ถูกสร้างใหม่เอง
    trk, n_frames, t0 = {}, 0, time.perf_counter()
    for res in MODEL.track(source=SOURCE, stream=True, persist=False, tracker=M["tracker"], conf=conf, iou=M["iou"], nms=nms,
                           imgsz=M["imgsz"], device=DEVICE, verbose=False):
        if res.boxes is not None and len(res.boxes) and res.boxes.id is not None:
            for box, c, cf, tid in zip(res.boxes.xyxy.tolist(), res.boxes.cls.int().tolist(), res.boxes.conf.tolist(), res.boxes.id.int().tolist()):
                tr = trk.setdefault(tid, new_track(0))
                update_track(tr, box, c, cf, res.orig_shape, 0)
        n_frames += 1
        if max_frames and n_frames >= max_frames:
            break
    return trk, n_frames, time.perf_counter() - t0, tuple(res.orig_shape)

ab_rows = []
if is_sim and EXPECTED and AB["enabled"]:
    exp_total = sum(EXPECTED.values())
    for nms_v in AB["nms"]:
        for conf_v in AB["conf"]:
            trk, nf, sec, shape = run_tracks(nms_v, conf_v, max_frames=400 if QUICK_TEST else None)
            for rule in AB["rule"]:
                for mf in AB["min_track_frames"]:
                    counts = Counter(NAMES[track_class(tr)] for tr in trk.values() if track_countable(tr, rule, mf, shape))
                    err = {n: counts.get(n, 0) - EXPECTED[n] for n in NAMES}
                    ab_rows.append({"nms": "NMS" if nms_v is None else "NMS-free", "conf": conf_v, "rule": rule, "min_track_frames": mf,
                                    **{f"count_{n}": counts.get(n, 0) for n in NAMES}, "total": sum(counts.values()), "expected": exp_total,
                                    "total_err": sum(counts.values()) - exp_total, "abs_err_sum": sum(abs(v) for v in err.values()),
                                    "tracks_total": len(trk), "frames": nf, "sec": round(sec, 1)})
            life = [tr["frames"] for tr in trk.values()]
            n_r = len(AB["rule"]) * len(AB["min_track_frames"])
            print(f"  nms={'NMS' if nms_v is None else 'NMS-free':<8} conf={conf_v:.3f} | tracks {len(trk):3d} (อายุ median {np.median(life):.0f} เฟรม) | ควร {exp_total} | {sec:.1f} s")
            for r in ab_rows[-n_r:]:
                print(f"      {r['rule']:<10} min{r['min_track_frames']:<3d}: {r['total']:2d} ({r['total_err']:+d})  |err| {r['abs_err_sum']}")
    ab = pd.DataFrame(ab_rows)
    ab["total_err_abs"] = ab.total_err.abs()
    ab = ab.sort_values(["abs_err_sum", "total_err_abs", "min_track_frames"], kind="stable").drop(columns="total_err_abs").reset_index(drop=True)
    ab.to_csv(DIRS["reports"] / "07_counting_ablation.csv", index=False, encoding="utf-8-sig")
    print("\n— ablation (เรียงจาก |error| รวมต่อคลาสน้อยไปมาก; เสมอกัน → กฎที่อยู่ก่อนใน list / min_track_frames น้อยกว่า) —")
    print(ab[["nms", "conf", "rule", "min_track_frames"] + [f"count_{n}" for n in NAMES] + ["total", "expected", "total_err", "abs_err_sum", "tracks_total"]].head(12).to_string(index=False))
    best = ab.iloc[0]
    RUN["ablation"] = {"rows": ab_rows, "best": {"nms": None if best.nms == "NMS" else False, "conf": float(best.conf), "rule": str(best.rule), "min_track_frames": int(best.min_track_frames),
                                                 "abs_err_sum": int(best.abs_err_sum), "total_err": int(best.total_err)}}
    if AB["auto_select"]:
        NB07["model"]["nms"] = None if best.nms == "NMS" else False
        NB07["model"]["conf"] = float(best.conf)
        NB07["count"]["rule"] = str(best.rule)
        NB07["count"]["min_track_frames"] = int(best.min_track_frames)
        print(f"\n[SELECT] ใช้ nms={best.nms} conf={best.conf:.3f} rule={best.rule} min_track_frames={int(best.min_track_frames)} → นับ {int(best.total)}/{exp_total} "
              f"(|error| รวมต่อคลาส {int(best.abs_err_sum)}) กับวิดีโอจริงใน CELL 4")
    # กราฟ: |error| รวมต่อคลาส ของทุก config
    fig, ax = plt.subplots(figsize=(9.5, 3.8))
    gcols = ["nms", "conf", "rule"]
    labels = [f"{r.nms}\n{r.rule}" + (f"\nconf {r.conf:.2f}" if len(AB["conf"]) > 1 else "") for _, r in ab.drop_duplicates(gcols).iterrows()]
    groups = list(ab.drop_duplicates(gcols)[gcols].itertuples(index=False, name=None))
    x = np.arange(len(groups))
    wbar = 0.8 / len(AB["min_track_frames"])
    for j, mf in enumerate(AB["min_track_frames"]):
        vals = [int(ab[(ab.nms == g[0]) & (ab.conf == g[1]) & (ab.rule == g[2]) & (ab.min_track_frames == mf)].abs_err_sum.iloc[0]) for g in groups]
        bars = ax.bar(x - 0.4 + wbar * (j + 0.5), vals, width=wbar * 0.92, label=f"min_track_frames = {mf}", color=["#2a78d6", "#eb6834", "#7a7a7a", "#2a9d8f", "#8e6bbf"][j % 5])
        for b_, v in zip(bars, vals):
            ax.text(b_.get_x() + b_.get_width() / 2, b_.get_height(), str(v), ha="center", va="bottom", fontsize=8, color=viz.TEXT["secondary"])
    ax.set_xticks(x, labels, fontsize=9)
    ax.set_ylabel("sum |count − expected| over classes")
    ax.set_title(f"Counting ablation on simulated conveyor (expected {exp_total} bananas)", loc="left", fontsize=11)
    ax.margins(y=0.2)
    ax.legend(fontsize=8)
    fig.tight_layout()
    out = viz.save_fig(fig, DIRS["figures"], "07_counting_ablation.png")
    plt.show()
    plt.close(fig)
    print(f"[OK] reports/07_counting_ablation.csv | {out.relative_to(PROJECT_ROOT)}")
else:
    print("[SKIP] ablation ต้องใช้วิดีโอจำลอง (มี ground truth) — ใช้ค่า nms/conf/min_track_frames จาก CELL 1 ตามที่ตั้งไว้")


In [ ]:
# === CELL 4 === Run: ByteTrack + วาดผล + overlay สด + QC alert → เขียน .mp4 + เก็บ log ทุกเฟรม
# ทำอะไร : model.track(stream=True, persist=False) → ต่อเฟรม: อัปเดตประวัติ track (คลาส/conf), ยืนยัน track ที่เห็น ≥ min_track_frames,
#          นับต่อคลาสจาก majority vote, ตรวจ QC ของผลิตภัณฑ์ที่เลือก, เตือน overripe เกิน threshold, วาดด้วย PIL (ไทย) แล้วเขียนวิดีโอ
# ทำไม   : spec ข้อ 10 ทั้งหมด — และ track id คือกลไกที่ทำให้กล้วยผลเดิมไม่ถูกนับซ้ำเมื่อเคลื่อนผ่านหลายเฟรม
M = NB07["model"]
QC, CNT, OV = NB07["qc"], NB07["count"], NB07["overlay"]
PRODUCT = QC["active_product"]
PASS_SET = set(QC["products"][PRODUCT]["pass"])
src_is_file = isinstance(SOURCE, str) and Path(SOURCE).is_file()
src_fps = 30.0
if src_is_file:
    cap = cv2.VideoCapture(str(SOURCE))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
else:
    n_total = None
stem = Path(str(SOURCE)).stem if isinstance(SOURCE, str) else f"camera{SOURCE}"
OUT_VIDEO = DIRS["videos_out"] / f"{stem}_annotated.mp4"
DIRS["videos_out"].mkdir(parents=True, exist_ok=True)

tracks = {}                       # track_id → {"classes": Counter(conf-weighted), "frames": n, "first": frame, "last": frame, "confirmed_at": frame|None}
confirmed_order = []              # ลำดับ track ที่ถูกยืนยัน (ใช้ทำ rolling window ของการเตือน)
log_rows, alerts, sample_frames = [], [], {}   # alerts = episode ต่อเนื่อง {frame_start, frame_end, t_start, t_end, peak_share}
writer = None
t_start = time.perf_counter()
fps_hist = deque(maxlen=30)

def live_counts():
    c = Counter()
    for tid, tr in tracks.items():
        if tr["confirmed_at"] is not None:
            c[track_class(tr)] += 1
    return c

_font_cache = {}
def fonts_for(h):
    # ฟอนต์/ความหนาเส้นตามความสูงเฟรม (วิดีโอ 360p จะได้ตัวหนังสือครึ่งหนึ่งของ 720p) — โหลดครั้งเดียวต่อขนาด
    sc = max(0.4, h / OV.get("ref_height", 720)) * OV.get("scale", 1.0)
    key = round(sc, 2)
    if key not in _font_cache:
        _font_cache[key] = (load_font(max(10, round(OV["font_size"] * sc))), load_font(max(9, round(OV["box_font_size"] * sc))),
                            load_font(max(12, round(30 * sc))), max(1, round(3 * sc)))
    return _font_cache[key]

def draw_frame(img_bgr, res, frame_idx, ts, counts, fps_now, alert_on):
    # วาดกล่อง+label (ไทย/อังกฤษ + conf + track id) และ panel สถิติสด ด้วย PIL
    im = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)).convert("RGBA")
    d = ImageDraw.Draw(im, "RGBA")
    FONT, FONT_BOX, FONT_BIG, LW = fonts_for(im.height)
    if CNT["rule"] == "cross_line":                            # เส้นนับ (เส้นประ) ให้เห็นว่าผลถูกนับตรงไหน
        if CNT["line_axis"] == "x":
            lx = int(CNT["line_x"] * im.width)
            for yy in range(0, im.height, 24):
                d.line([(lx, yy), (lx, min(yy + 12, im.height))], fill=(255, 255, 255, 150), width=2)
        else:
            ly = int(CNT["line_x"] * im.height)
            for xx in range(0, im.width, 24):
                d.line([(xx, ly), (min(xx + 12, im.width), ly)], fill=(255, 255, 255, 150), width=2)
    if res.boxes is not None and len(res.boxes):
        ids = res.boxes.id.int().tolist() if res.boxes.id is not None else [None] * len(res.boxes)
        placed = []                                   # label ที่วางแล้ว (x1,y1,x2,y2) — กันวางทับกัน
        order = sorted(zip(res.boxes.xyxy.tolist(), res.boxes.cls.int().tolist(), res.boxes.conf.tolist(), ids), key=lambda r: (r[0][1], r[0][0]))
        for (x1, y1, x2, y2), c, cf, tid in order:
            d.rectangle([x1, y1, x2, y2], outline=RGB[c] + (255,), width=LW)
        for (x1, y1, x2, y2), c, cf, tid in order:   # วาด label หลังกล่องทั้งหมด → label ไม่ถูกเส้นกล่องอื่นทับ
            col = RGB[c]
            label = f"{NAMES[c]} · {TH[c]} {cf:.2f}" if THAI_OK else f"{NAMES[c]} {cf:.2f}"
            if tid is not None:
                label += f"  #{tid}"
            tw, th_ = d.textbbox((0, 0), label, font=FONT_BOX)[2:]
            lw, lh = tw + 8, th_ + 6
            lx = min(max(0, x1), im.width - lw)
            cands = [y1 - lh - 2, y1 + 2, y2 + 2]     # เหนือกล่อง → ในกล่อง → ใต้กล่อง
            cands += [y1 + 2 + k * (lh + 2) for k in range(1, 6)]   # ถ้ายังชน: ไล่ลงในกล่องทีละแถว
            def hit(y):
                return y < 0 or y + lh > im.height or any(not (lx + lw <= px1 or lx >= px2 or y + lh <= py1 or y >= py2) for px1, py1, px2, py2 in placed)
            ty = next((y for y in cands if not hit(y)), y1 + 2)
            placed.append((lx, ty, lx + lw, ty + lh))
            d.rectangle([lx, ty, lx + lw, ty + lh], fill=col + (230,))
            d.text((lx + 4, ty + 2), label, font=FONT_BOX, fill=(0, 0, 0, 255) if c in (1, 2) else (255, 255, 255, 255))
    # panel (กะทัดรัด: 6 บรรทัด, ระยะห่างทุกอย่างสเกลตามฟอนต์)
    total = sum(counts.values())
    n_pass = sum(counts.get(c, 0) for c in CLASSES if NAMES[c] in PASS_SET)
    th_short = QC["products"][PRODUCT]["th"].split(" (")[0] if THAI_OK else ""
    lines = [f"frame {frame_idx}  t={ts:.1f}s  FPS {fps_now:.0f}  {'นับแล้ว' if THAI_OK else 'counted'} {total}"]
    for c in CLASSES:
        n = counts.get(c, 0)
        lines.append(f"{NAMES[c]}{' ' + TH[c] if THAI_OK else ''}: {n} ({n / total * 100 if total else 0:.0f}%) {'ผ่าน' if NAMES[c] in PASS_SET else 'ไม่ผ่าน'}")
    lines.append(f"QC {PRODUCT}{' ' + th_short if th_short else ''}: PASS {n_pass}  FAIL {total - n_pass}")
    if OV.get("panel", True):
        fs = FONT.size
        pad, gap, dot = round(fs * 0.5), round(fs * 0.3), round(fs * 0.55)
        indent = dot + round(fs * 0.4)                         # บรรทัดคลาส: จุดสี + เว้นระยะ แล้วค่อยเป็นข้อความ
        pw = max(d.textbbox((0, 0), l, font=FONT)[2] + (indent if 1 <= i <= NC else 0) for i, l in enumerate(lines)) + pad * 2
        ph = (fs + gap) * len(lines) + pad * 2 - gap
        d.rectangle([8, 8, 8 + pw, 8 + ph], fill=(0, 0, 0, int(255 * OV["panel_alpha"])))
        y = 8 + pad
        for i, l in enumerate(lines):
            x = 8 + pad
            if 1 <= i <= NC:
                c = list(CLASSES)[i - 1]
                d.ellipse([x, y + (fs - dot) // 2 + 1, x + dot, y + (fs - dot) // 2 + 1 + dot], fill=RGB[c] + (255,))
                x += indent
            d.text((x, y), l, font=FONT, fill=(255, 255, 255, 255))
            y += fs + gap
    if alert_on:
        msg = f"ALERT: overripe > {QC['overripe_alert_share'] * 100:.0f}% {'ใน ' + str(QC['overripe_alert_window']) + ' ผลล่าสุด' if THAI_OK else 'in last ' + str(QC['overripe_alert_window'])}"
        tw, th_ = d.textbbox((0, 0), msg, font=FONT_BIG)[2:]
        W_, pad_b = im.width, round(FONT_BIG.size * 0.4)
        d.rectangle([W_ - tw - pad_b * 2 - 8, 8, W_ - 8, 8 + th_ + pad_b * 2], fill=(200, 30, 30, 235))
        d.text((W_ - tw - pad_b - 8, 8 + pad_b), msg, font=FONT_BIG, fill=(255, 255, 255, 255))
    return cv2.cvtColor(np.asarray(im.convert("RGB")), cv2.COLOR_RGB2BGR)

frame_idx = 0
results = MODEL.track(source=SOURCE, stream=True, persist=False, tracker=M["tracker"], conf=M["conf"], iou=M["iou"], nms=M["nms"],
                      imgsz=M["imgsz"], device=DEVICE, verbose=False)
for res in results:
    ts = frame_idx / src_fps if src_is_file else time.perf_counter() - t_start
    if res.boxes is not None and len(res.boxes) and res.boxes.id is not None:
        for (x1, y1, x2, y2), c, cf, tid in zip(res.boxes.xyxy.tolist(), res.boxes.cls.int().tolist(), res.boxes.conf.tolist(), res.boxes.id.int().tolist()):
            H_, W_ = res.orig_shape
            tr = tracks.setdefault(tid, new_track(frame_idx))
            update_track(tr, (x1, y1, x2, y2), c, cf, res.orig_shape, frame_idx)
            if tr["confirmed_at"] is None and track_countable(tr, CNT["rule"], CNT["min_track_frames"], res.orig_shape):
                tr["confirmed_at"] = frame_idx                     # cross_line: นับ ณ เฟรมที่ผ่านเส้น (เหมือน gate จริง)
                confirmed_order.append(tid)
            log_rows.append({"frame": frame_idx, "timestamp_s": round(ts, 3), "track_id": tid, "class_id": c, "class": NAMES[c], "confidence": round(cf, 4),
                             "x1": round(x1, 1), "y1": round(y1, 1), "x2": round(x2, 1), "y2": round(y2, 1),
                             "xc_n": round((x1 + x2) / 2 / W_, 4), "yc_n": round((y1 + y2) / 2 / H_, 4), "w_n": round((x2 - x1) / W_, 4), "h_n": round((y2 - y1) / H_, 4)})
    counts = live_counts()
    # QC alert: สัดส่วน overripe ใน window ล่าสุด
    recent = confirmed_order[-QC["overripe_alert_window"]:]
    n_recent_over = sum(1 for tid in recent if NAMES[track_class(tracks[tid])] == "overripe")
    alert_on = len(confirmed_order) >= QC["alert_min_count"] and len(recent) > 0 and n_recent_over / len(recent) > QC["overripe_alert_share"]
    if alert_on:                                   # episode = ช่วงที่เงื่อนไขเป็นจริงต่อเนื่อง (ไม่นับ re-trigger ซ้ำ)
        share = round(n_recent_over / len(recent), 3)
        if alerts and alerts[-1]["frame_end"] == frame_idx - 1:
            alerts[-1].update(frame_end=frame_idx, t_end=round(ts, 2), peak_share=max(alerts[-1]["peak_share"], share))
        else:
            alerts.append({"frame_start": frame_idx, "frame_end": frame_idx, "t_start": round(ts, 2), "t_end": round(ts, 2), "peak_share": share, "window": len(recent)})
    fps_hist.append(1.0 / max(sum(res.speed.values()) / 1000, 1e-6))
    fps_now = float(np.mean(fps_hist))
    frame = draw_frame(res.orig_img, res, frame_idx, ts, counts, fps_now, alert_on)
    if writer is None:
        writer = cv2.VideoWriter(str(OUT_VIDEO), cv2.VideoWriter_fourcc(*"mp4v"), src_fps, (frame.shape[1], frame.shape[0]))
    writer.write(frame)
    if n_total and frame_idx in {int(n_total * q) for q in (0.15, 0.4, 0.65, 0.9)}:
        sample_frames[frame_idx] = frame.copy()
    if frame_idx % 200 == 0:
        print(f"  frame {frame_idx}{'/' + str(n_total) if n_total else ''} | tracks {len(tracks)} | counted {sum(counts.values())} | model FPS {fps_now:.0f}")
    frame_idx += 1
    if QUICK_TEST and frame_idx >= 400:
        break
if writer is not None:
    writer.release()
wall = time.perf_counter() - t_start
for ep in alerts:
    ep["duration_s"] = round(ep["t_end"] - ep["t_start"] + 1 / src_fps, 2)
final_counts = {NAMES[c]: n for c, n in sorted(live_counts().items())}
print(f"\n[OK] {frame_idx} เฟรม ใน {wall:.1f} s (pipeline {frame_idx / wall:.1f} FPS รวมวาด/เขียนวิดีโอ | model {np.mean(fps_hist):.0f} FPS) → {OUT_VIDEO.relative_to(PROJECT_ROOT)}")
print(f"[OK] config: nms={'NMS' if M['nms'] is None else 'NMS-free'} conf={M['conf']:.3f} rule={CNT['rule']} min_track_frames={CNT['min_track_frames']}")
print(f"[OK] tracks ทั้งหมด {len(tracks)} | ยืนยัน (≥{CNT['min_track_frames']} เฟรม) {len(confirmed_order)} | นับได้: {final_counts} | alert {len(alerts)} episode "
      f"รวม {sum(ep['duration_s'] for ep in alerts):.1f} s")


In [ ]:
# === CELL 5 === Export detection_log.csv + summary_report.csv + เทียบกับ ground truth (วิดีโอจำลอง) + รูปตัวอย่างเฟรม
# ทำอะไร : เขียน log ทุก detection (frame, timestamp, track_id, class, confidence, bbox) และสรุปต่อคลาส/QC/alert;
#          ถ้าเป็นวิดีโอจำลอง → counting error ต่อคลาส; บันทึกเฟรมตัวอย่าง 4 ช่วง + กราฟนับได้ vs ควรนับ
# ทำไม   : spec ข้อ 10 export ครบ + วิดีโอใส่เล่มรายงานไม่ได้ → ต้องมีรูปนิ่งและตัวเลข
LOG_COLS = ["frame", "timestamp_s", "track_id", "class_id", "class", "confidence", "x1", "y1", "x2", "y2", "xc_n", "yc_n", "w_n", "h_n"]
log_df = pd.DataFrame(log_rows, columns=LOG_COLS)
log_path = DIRS["reports"] / "detection_log.csv"
log_df.to_csv(log_path, index=False, encoding="utf-8-sig")

rows = []
total = sum(final_counts.values())
for c in CLASSES:
    n = final_counts.get(NAMES[c], 0)
    tids = [tid for tid, tr in tracks.items() if tr["confirmed_at"] is not None and track_class(tr) == c]
    mean_conf = float(log_df[log_df.track_id.isin(tids)]["confidence"].mean()) if tids and len(log_df) else float("nan")
    rows.append({"class": NAMES[c], "class_th": TH[c], "count": n, "share_pct": round(n / total * 100, 1) if total else 0.0, "mean_conf": round(mean_conf, 3),
                 "qc_status": "PASS" if NAMES[c] in PASS_SET else "FAIL", "expected": (EXPECTED or {}).get(NAMES[c]),
                 "count_error": (n - EXPECTED[NAMES[c]]) if EXPECTED else None})
summary = pd.DataFrame(rows)
n_pass = int(summary[summary.qc_status == "PASS"]["count"].sum())
extra = pd.DataFrame([
    {"class": "TOTAL", "count": total, "share_pct": 100.0, "qc_status": f"PASS {n_pass} / FAIL {total - n_pass}", "expected": sum(EXPECTED.values()) if EXPECTED else None,
     "count_error": (total - sum(EXPECTED.values())) if EXPECTED else None},
    {"class": "product", "qc_status": PRODUCT}, {"class": "alerts", "count": len(alerts), "qc_status": "; ".join(f"t={a['t_start']}–{a['t_end']}s peak {a['peak_share'] * 100:.0f}%" for a in alerts[:10])},
    {"class": "frames", "count": frame_idx, "qc_status": f"{frame_idx / wall:.1f} FPS pipeline"},
])
summary_all = pd.concat([summary, extra], ignore_index=True)
summary_path = DIRS["reports"] / "summary_report.csv"
summary_all.to_csv(summary_path, index=False, encoding="utf-8-sig")
print("— summary —")
print(summary.to_string(index=False))
print(f"\nTOTAL {total} | QC {PRODUCT}: PASS {n_pass} / FAIL {total - n_pass} | alerts {len(alerts)}")
if EXPECTED:
    exp_total = sum(EXPECTED.values())
    mae = float(np.mean([abs(r["count_error"]) for r in rows]))
    print(f"[COUNT CHECK] วิดีโอจำลอง: ควรนับ {exp_total} นับได้ {total} (error {total - exp_total:+d} = {abs(total - exp_total) / max(exp_total, 1) * 100:.1f}%) | MAE ต่อคลาส {mae:.2f} ผล")
print(f"[OK] {log_path.relative_to(PROJECT_ROOT)} ({len(log_df):,} แถว) | {summary_path.relative_to(PROJECT_ROOT)}")

# --- ตรวจต่อภาพ (วิดีโอจำลอง): track ไหนอยู่บนภาพไหน → เทียบคลาสที่นับได้กับ label จริงของภาพนั้น ---
# ทำไม : ตัวเลขรวมต่อคลาสอาจ "บังเอิญ" ตรง (ผิด 2 ทางหักกัน) — ต้องดูรายภาพว่าโมเดลทายคลาสตรง label หรือไม่
if SIM_META and "tiles" in SIM_META and len(log_df):
    spd = SIM_META["settings"]["speed_px"]
    log_df["belt_x"] = (log_df.x1 + log_df.x2) / 2 + log_df.frame * spd      # พิกัด x บนสายพาน (นิ่ง) ของกล่อง
    tile_of = {}
    for tid, g in log_df.groupby("track_id"):
        if tracks[tid]["confirmed_at"] is None:
            continue
        bx = float(g.belt_x.median())
        k = next((i for i, tm in enumerate(SIM_META["tiles"]) if tm["start"] - 20 <= bx < tm["start"] + tm["width"] + 20), None)
        tile_of[tid] = k
    per_img = []
    for i, tm in enumerate(SIM_META["tiles"]):
        pred = Counter(NAMES[track_class(tracks[t])] for t, k in tile_of.items() if k == i)
        gt = tm["gt"]
        n_gt, n_pred = sum(gt.values()), sum(pred.values())
        gt_s = ", ".join(f"{n}×{v}" for n, v in gt.items() if v)
        pred_s = ", ".join(f"{n}×{v}" for n, v in pred.items() if v) or "-"
        if pred == Counter({n: v for n, v in gt.items() if v}):
            status = "OK"
        elif n_pred == 0:
            status = "missed"
        elif {n for n, v in pred.items() if v} == {n for n, v in gt.items() if v}:
            status = "count"
        else:
            status = "class"
        per_img.append({"order": i + 1, "image": tm["image"], "source": tm["source"], "gt": gt_s, "n_gt": n_gt, "predicted": pred_s, "n_pred": n_pred, "status": status,
                        "track_ids": ",".join(str(t) for t, k in tile_of.items() if k == i)})
    per_img = pd.DataFrame(per_img)
    per_img.to_csv(DIRS["reports"] / "07_per_image_check.csv", index=False, encoding="utf-8-sig")
    n_ok = int((per_img.status == "OK").sum())
    print(f"\n[PER-IMAGE CHECK] ภาพที่คลาส+จำนวนตรง label: {n_ok}/{len(per_img)} | ผิดจำนวน {int((per_img.status == 'count').sum())} | ผิดคลาส {int((per_img.status == 'class').sum())} | ไม่เจอเลย {int((per_img.status == 'missed').sum())} "
          f"| track ที่ไม่ตกบนภาพใด {sum(1 for k in tile_of.values() if k is None)}")
    bad = per_img[per_img.status != "OK"]
    if len(bad):
        print("  ภาพที่ไม่ตรง (status: count = จำนวนผิด, class = คลาสผิด, missed = ไม่เจอเลย):")
        print(bad[["order", "image", "source", "gt", "predicted", "status"]].to_string(index=False))
    print(f"[OK] reports/07_per_image_check.csv")

    # --- รูปเคสที่ไม่ตรง: เฟรมที่ภาพนั้นอยู่กลางสายพาน — GT (ขาวหนา) vs กล่อง track ในเฟรมนั้น (สีคลาส + #id) ---
    # ทำไม : เขียน failure analysis ในเล่มได้จากรูป (กล่องซ้ำ / แตก / คลาสผิด) ไม่ต้องไล่ดูวิดีโอ
    bad = per_img[per_img.status != "OK"].head(6)
    if len(bad):
        cap = cv2.VideoCapture(str(SOURCE))
        W_, H_ = SIM_META["canvas"]
        n_b = len(bad)
        cols = 2 if n_b > 1 else 1
        rws = math.ceil(n_b / cols)
        fig, axes = plt.subplots(rws, cols, figsize=(6.6 * cols, 4.0 * rws), squeeze=False)
        for ax in axes.reshape(-1):
            ax.axis("off")
        for ax, (_, r) in zip(axes.reshape(-1), bad.iterrows()):
            tm = SIM_META["tiles"][int(r.order) - 1]
            fi = int(round((tm["start"] + tm["width"] / 2 - W_ / 2) / spd))
            fi = max(0, min(fi, frame_idx - 1))
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ok, fr = cap.read()
            if not ok:
                continue
            im = Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
            d = ImageDraw.Draw(im)
            x0, y0 = tm["start"] - fi * spd, (H_ - tm["height"]) // 2
            for ln in (DIRS["datasets"] / "test" / "labels" / (Path(tm["image"]).stem + ".txt")).read_text().splitlines():
                if not ln.strip():
                    continue
                c_, xc, yc, w, h = ln.split()[:5]
                xc, yc, w, h = float(xc) * tm["width"], float(yc) * tm["height"], float(w) * tm["width"], float(h) * tm["height"]
                d.rectangle([x0 + xc - w / 2, y0 + yc - h / 2, x0 + xc + w / 2, y0 + yc + h / 2], outline=(255, 255, 255), width=6)
                d.text((x0 + xc - w / 2 + 6, y0 + yc - h / 2 + 4), f"GT {NAMES[int(c_)]}", font=FONT_BOX, fill=(255, 255, 255))
            for _, b in log_df[log_df.frame == fi].iterrows():
                col = RGB[int(b.class_id)]
                d.rectangle([b.x1, b.y1, b.x2, b.y2], outline=col, width=3)
                d.text((b.x1 + 4, b.y2 - FONT_BOX.size - 8), f"#{int(b.track_id)} {b['class']} {b.confidence:.2f}", font=FONT_BOX, fill=col)
            ax.imshow(im)
            ax.set_title(f"#{int(r.order)} {r.source} · GT {r['gt']} → counted {r['predicted']} ({r['status']}) · frame {fi}", loc="left", fontsize=9)
        cap.release()
        fig.suptitle("Mismatch cases — white = ground truth boxes, colour = tracked boxes at the frame where the image is centred", x=0.01, ha="left", fontsize=11)
        fig.tight_layout()
        out = viz.save_fig(fig, DIRS["figures"], "07_mismatch_cases.png")
        plt.show()
        plt.close(fig)
        print(f"[OK] {out.relative_to(PROJECT_ROOT)} ({n_b} ภาพ)")

# --- รูป: เฟรมตัวอย่าง 4 ช่วง + กราฟนับได้ vs ควรนับ ---
if sample_frames:
    fig, axes = plt.subplots(2, 2, figsize=(13, 7.6))
    for ax, (fi, fr) in zip(axes.reshape(-1), sorted(sample_frames.items())):
        ax.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        ax.axis("off")
        ax.set_title(f"frame {fi} (t = {fi / src_fps:.1f} s)", loc="left", fontsize=10)
    fig.suptitle(f"Video inference — {Path(str(SOURCE)).name} · ByteTrack · product = {PRODUCT}", x=0.01, ha="left", fontsize=12)
    fig.tight_layout()
    out = viz.save_fig(fig, DIRS["figures"], "07_video_frames.png")
    plt.show()
    plt.close(fig)
    print(f"[OK] {out.relative_to(PROJECT_ROOT)}")
fig, ax = plt.subplots(figsize=(7.5, 3.8))
x = np.arange(NC)
if EXPECTED:
    b1 = ax.bar(x - 0.19, [EXPECTED[n] for n in NAMES], width=0.36, color="#b9b8b1", label="expected (ground truth)")
    for b_ in b1:
        ax.text(b_.get_x() + b_.get_width() / 2, b_.get_height(), f"{int(b_.get_height())}", ha="center", va="bottom", fontsize=8, color=viz.TEXT["secondary"])
b2 = ax.bar(x + (0.19 if EXPECTED else 0), [final_counts.get(n, 0) for n in NAMES], width=0.36, color=[viz.class_colors(CONFIG)[c] for c in CLASSES], label="counted (tracks)")
for b_ in b2:
    ax.text(b_.get_x() + b_.get_width() / 2, b_.get_height(), f"{int(b_.get_height())}", ha="center", va="bottom", fontsize=8, color=viz.TEXT["secondary"])
ax.set_xticks(x, NAMES)
ax.set_title("Bananas counted per class" + (" vs expected (simulated conveyor)" if EXPECTED else ""), loc="left", fontsize=11)
ax.margins(y=0.15)
ax.legend(fontsize=8)
fig.tight_layout()
out = viz.save_fig(fig, DIRS["figures"], "07_counts_vs_expected.png")
plt.show()
plt.close(fig)
print(f"[OK] {out.relative_to(PROJECT_ROOT)}")


In [ ]:
# === CELL 6 === สรุป QC decision + บันทึก reports/07_video_summary.json + วิธีสลับไปกล้องจริง
# ทำอะไร : รวมผลของ notebook นี้ลง JSON (ตั้งค่า, จำนวน, alert, ความเร็ว) + พิมพ์ QC decision ต่อผลิตภัณฑ์ทั้ง 3 จากผลนับเดียวกัน
# ทำไม   : ผู้ใช้เปลี่ยนผลิตภัณฑ์ได้โดยไม่ต้องรันใหม่ (นับครั้งเดียว ตัดสินได้หลายเกณฑ์) และเล่มรายงานอ่านตัวเลขจากไฟล์นี้
print("— QC decision ต่อผลิตภัณฑ์ (จากจำนวนที่นับได้ชุดเดียวกัน) —")
qc_table = []
for prod, spec in NB07["qc"]["products"].items():
    p = sum(final_counts.get(n, 0) for n in spec["pass"])
    qc_table.append({"product": prod, "th": spec["th"], "pass_classes": ",".join(spec["pass"]), "pass": p, "fail": total - p, "pass_pct": round(p / total * 100, 1) if total else 0.0})
    print(f"  {prod:<13} ({spec['th']}): ผ่าน {p:3d} / ไม่ผ่าน {total - p:3d}  ({qc_table[-1]['pass_pct']}% ผ่าน) — เกณฑ์ผ่าน: {spec['pass']}")
over_share = final_counts.get("overripe", 0) / total if total else 0.0
print(f"\n  overripe ทั้งวิดีโอ {over_share * 100:.1f}% | เกณฑ์เตือน {NB07['qc']['overripe_alert_share'] * 100:.0f}% ใน {NB07['qc']['overripe_alert_window']} ผลล่าสุด → alert {len(alerts)} episode "
      f"รวม {sum(ep['duration_s'] for ep in alerts):.1f} s" + (f" (แรกสุด t={alerts[0]['t_start']}–{alerts[0]['t_end']}s peak {alerts[0]['peak_share'] * 100:.0f}%)" if alerts else ""))
for ep in alerts:
    print(f"    episode t={ep['t_start']:6.1f}–{ep['t_end']:6.1f}s ({ep['duration_s']:.1f} s) peak overripe {ep['peak_share'] * 100:.0f}% ใน {ep['window']} ผลล่าสุด")
RUN["settings"] = NB07                                   # ค่าที่ใช้จริง (หลัง auto-select ใน CELL 3)
RUN.update({"output_video": str(OUT_VIDEO.relative_to(PROJECT_ROOT)), "frames": frame_idx, "pipeline_fps": round(frame_idx / wall, 1), "model_fps": round(float(np.mean(fps_hist)), 1),
            "tracks_total": len(tracks), "tracks_confirmed": len(confirmed_order), "counts": final_counts, "expected": EXPECTED, "qc": qc_table, "alerts": alerts,
            "thai_font": THAI_OK, "finished_at": datetime.now().isoformat(timespec="seconds")})
with open(DIRS["reports"] / "07_video_summary.json", "w", encoding="utf-8") as f:
    json.dump(RUN, f, ensure_ascii=False, indent=2, default=str)
CONFIG["video_inference"] = {"source": str(SOURCE), "output": RUN["output_video"], "tracker": NB07["model"]["tracker"], "conf": NB07["model"]["conf"], "nms": NB07["model"]["nms"],
                             "rule": NB07["count"]["rule"], "min_track_frames": NB07["count"]["min_track_frames"], "selected_by": "ablation" if RUN.get("ablation") and NB07["ablation"]["auto_select"] else "manual",
                             "active_product": PRODUCT, "counts": final_counts, "expected": EXPECTED, "updated_by": "07_video_inference.ipynb"}
with open(CFG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, allow_unicode=True, sort_keys=False)
print(f"\n[OK] reports/07_video_summary.json | project.yaml → video_inference")
print("\n=== SUMMARY · Notebook 07 ===")
print(f"  วิดีโอ: {RUN['output_video']} | {frame_idx} เฟรม | pipeline {RUN['pipeline_fps']} FPS (model {RUN['model_fps']} FPS)")
print(f"  config: nms={'NMS' if NB07['model']['nms'] is None else 'NMS-free'} conf={NB07['model']['conf']:.3f} rule={NB07['count']['rule']} min_track_frames={NB07['count']['min_track_frames']}"
      + (" (เลือกจาก ablation CELL 3)" if RUN.get("ablation") and NB07["ablation"]["auto_select"] else ""))
print(f"  นับได้ {total} ผล: {final_counts}" + (f" | ควรนับ {sum(EXPECTED.values())}" if EXPECTED else "") + f" | QC {PRODUCT}: PASS {n_pass} / FAIL {total - n_pass} | alert {len(alerts)} episode")
print("  สลับไปกล้องจริง: แก้ SOURCE ใน CELL 1 (ไฟล์ .mp4 / โฟลเดอร์ภาพ / 0 = webcam / rtsp://...) แล้ว Run All — CELL 2 จะข้ามการสร้างวิดีโอจำลองเอง")


## ผลลัพธ์ที่ควรเห็น
- CELL 2: `[OK] สร้าง videos/input/simulated_conveyor.mp4: 28–30 ภาพ → ~1,500–1,800 เฟรม | ควรนับได้: {...}`
- CELL 3: ตาราง ablation (2 รอบ tracking ≈ 1 นาที; rule × min_track_frames ประเมิน offline) + `[SELECT] ใช้ nms=... rule=... min_track_frames=...` + `figures/07_counting_ablation.png`
- CELL 4: บรรทัดความคืบหน้าทุก 200 เฟรม → `[OK] N เฟรม ... → videos/output/simulated_conveyor_annotated.mp4` + จำนวนที่นับได้ต่อคลาส
- CELL 5: ตาราง summary + `[COUNT CHECK] ควรนับ X นับได้ Y (error ±Z)` + `[PER-IMAGE CHECK]` + รูปเคสที่ไม่ตรง (`07_mismatch_cases.png`); รูป 4 เฟรมพร้อม overlay (ดูวิดีโอเต็มได้โดยดาวน์โหลด .mp4 จาก file browser)
- CELL 6: QC decision ทั้ง 3 ผลิตภัณฑ์ + alert episode

## ใช้วิดีโอของตัวเอง
1. อัปโหลดไฟล์ (.mp4 / .avi / .mov) ไว้ที่ `videos/input/` (JupyterLab: เข้าโฟลเดอร์ → ปุ่ม Upload)
2. Run All → CELL 1 แสดงรายการไฟล์และช่องพิมพ์ใต้ cell → พิมพ์หมายเลข → ตอบว่ากล้วยเคลื่อนที่ทางไหน (1 ซ้าย↔ขวา / 2 บน↔ล่าง / 3 อยู่นิ่ง)
3. ผลอยู่ที่ `videos/output/<ชื่อไฟล์>_annotated.mp4`, `reports/detection_log.csv`, `reports/summary_report.csv`, `figures/07_video_frames.png`
   (ไม่มี `[COUNT CHECK]` เพราะไม่รู้จำนวนจริง — นับด้วยตาแล้วเทียบกับ summary_report.csv)
- ข้อแนะนำการถ่าย (จากผล robustness nb06): กล้องตั้งฉากกับกล้วย, เลี่ยงแสงจ้า, ให้กล้วยทุกลูกผ่านกลางเฟรม; `.mov` HEVC เปิดไม่ได้ → `ffmpeg -i in.mov -c:v libx264 -crf 20 out.mp4`
- `SOURCE = 0` (webcam) ใช้บนเซิร์ฟเวอร์ไม่ได้ (ไม่มีกล้อง) — ใช้ไฟล์หรือ rtsp:// เท่านั้น

## ปัญหาที่อาจเจอ + วิธีแก้
| อาการ | สาเหตุ | วิธีแก้ |
|---|---|---|
| label ไทยเป็นสี่เหลี่ยม / `[WARN] ไม่มีฟอนต์ไทย` | ดาวน์โหลด Sarabun ไม่ได้และไม่มีฟอนต์ไทยในระบบ | วางไฟล์ .ttf ที่รองรับไทยไว้ที่ `fonts/Sarabun-Regular.ttf` แล้ว Run All |
| นับได้มากกว่าควรนับ | กล่องซ้ำบนกล้วยที่ซ้อนกัน (NMS-free) หรือ track แตกเป็น 2 | ดูตาราง ablation CELL 3 — ใช้ nms=None และ/หรือเพิ่ม `min_track_frames`; ถ้ายังแตก เพิ่ม `track_buffer` ใน bytetrack.yaml (copy จาก ultralytics/cfg/trackers แล้วชี้ path) |
| นับได้น้อยกว่าควรนับ | กล้วยซ้อนกันจน detection รวมเป็นกล่องเดียว | ดู occlusion result ใน nb06 — ข้อจำกัดของโมเดล ไม่ใช่ tracker |
| วิดีโอ .mp4 เปิดในเบราว์เซอร์ไม่ได้ | codec mp4v | ดาวน์โหลดไปเปิดด้วย VLC หรือแปลงด้วย `ffmpeg -i in.mp4 -c:v libx264 out.mp4` |
| `ModuleNotFoundError: lap` | ByteTrack ต้องใช้ lap | `pip install lap` ใน kernel นี้ (nb01 ติดตั้งไว้แล้วตามปกติ) |
